In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

#my client id
client_id = '9692aa73-2ddc-4011-84eb-da302031b0ba'

#define an endpoint and parameters for the API request
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN17850',
    'elements': 'mean(air_temperature P1D)',
    'referencetime': '2025-01-01/2025-12-31',
}

#issue a HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id, ''))

#Extract JSON data from the response
weather_data = r.json()


In [ ]:
#create a pandas DataFrame from the JSON data
df = pd.DataFrame()
for i in range(len(weather_data['data'])):
    observations = pd.DataFrame(weather_data['data'][i]['observations'])
    observations['date'] = weather_data['data'][i]['referenceTime']

    df = pd.concat([df, observations], ignore_index=True)

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')


In [ ]:
#create a line plot of the mean air temperature over time
plt.figure(figsize=(12, 5))

plt.plot(df['date'], df['value'], color='blue', label='Mean Air Temperature')

plt.title('Daily Mean Air Temperature in ÅS (2025)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')

plt.legend()
plt.show()

In [ ]:
#create a summary table of the temperature statistics
summary = pd.DataFrame({
    'Statistic': ['Mean', 'Median', 'Minimum', 'Maximum'],
    'Temperature (°C)': [
        df['value'].mean(),
        df['value'].median(),
        df['value'].min(),
        df['value'].max()
    ]
})
print(summary)


In [ ]:
#Task 4 starts here 

import yaml
#create a function for counting heatwaves in 2025
def count_heatwaves(station_id, year, station_name):

    parameters = {    
        'sources': station_id,
        'elements': 'max(air_temperature P1D)',
        'referencetime': f'{year}-01-01/{year}-12-31',
    }

    r = requests.get(endpoint, parameters, auth=(client_id, ''))
    r.raise_for_status()
    heatwave_data = r.json()

    #create a DataFrame to store the heatwave data
    df_heatwave = pd.DataFrame()
    for observations in heatwave_data['data']:
        daily_data = pd.DataFrame(observations['observations'])
        daily_data['date'] = observations['referenceTime']
        df_heatwave = pd.concat([df_heatwave, daily_data], ignore_index=True)

    #convert the 'date' column to datetime and sort the DataFrame by date
    df_heatwave['date'] = pd.to_datetime(df_heatwave['date']).dt.date
    df_heatwave = df_heatwave.sort_values('date')

    #count the number of heatwaves (5 or more consecutive days with max temperature >= 27°C)
    consecutive_days = 0
    heatwave_dates = [] 
    start_date = None 
    previous_date = None

    #loop through the DataFrame to count consecutive days with max temperature >= 27°C
    for date,temp in zip(df_heatwave['date'], df_heatwave['value']):
        if temp >= 27:
            if consecutive_days == 0:
                start_date = date

            consecutive_days += 1
        else:
            if consecutive_days >= 5:
                heatwave_dates.append(f'{start_date} - {previous_date}')

            consecutive_days = 0
        previous_date = date

    #check if the last sequence of consecutive days qualifies as a heatwave
    if consecutive_days >= 5:
        heatwave_dates.append(f'{start_date} - {previous_date}')

    summary = {
    'dates': heatwave_dates,
    'station_id': station_id,
    'station_name': station_name,
    'year': year
    }

    with open('heatwave_summary.yml', 'w', encoding='utf-8') as file:
        yaml.safe_dump(summary, file, allow_unicode=True, sort_keys=False)

    
    return summary




#call the function to count heatwaves in 2025 for station SN17850 (Ås)
count_heatwaves('SN17850', 2025, 'Ås')

{'dates': ['2025-07-11 - 2025-07-20', '2025-07-21 - 2025-07-25'],
 'station_id': 'SN17850',
 'station_name': 'Ås',
 'year': 2025}